In [1]:
import tensorrt
import tensorflow as tf
from tensorflow import keras
import numpy as np
from tensorflow.keras import layers
from keras.callbacks import History
import matplotlib.pyplot as plt
import os
from tensorflow.keras.optimizers import Adam
from plain_neural_network import*
from matplotlib import cm
from keras import backend as K

2024-05-24 00:42:01.611237: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())
import tensorflow as tf; print(tf.config.list_physical_devices('GPU'))

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 16729418918904189125
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 45610434560
locality {
  bus_id: 2
  numa_node: 1
  links {
  }
}
incarnation: 4889329804808532162
physical_device_desc: "device: 0, name: NVIDIA A40, pci bus id: 0000:41:00.0, compute capability: 8.6"
xla_global_id: 416903419
]
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2024-05-24 00:42:05.184624: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /device:GPU:0 with 43497 MB memory:  -> device: 0, name: NVIDIA A40, pci bus id: 0000:41:00.0, compute capability: 8.6


## Loading South Atlantic data

In [3]:
import glob
import random
input_dir = "/albedo/work/user/ssunar/for_paper/segmentation_masks/south_atlantic/"
input_file_paths = sorted(glob.glob(input_dir+'*.nc'))
# These data samples throw error
input_file_paths.pop(4)
input_file_paths.pop(11)
input_file_paths.pop(29)

print(len(input_file_paths))
random_num = random.randint(0,len(input_file_paths))

data = xr.open_mfdataset(input_file_paths[:],combine = 'nested', concat_dim="TIME").astype('float32')
X_south_atlantic = data.ssh.to_numpy()
test_y_south_atlantic = data.seg_mask.to_numpy()

X_south_atlantic = X_south_atlantic[:3500]
test_y_south_atlantic = test_y_south_atlantic[:3500]
X_south_atlantic[X_south_atlantic>1000] = 0

print(X_south_atlantic.shape)
print(test_y_south_atlantic.shape)

129
(3500, 840, 480)
(3500, 840, 480)


In [4]:
data_long = data.LONGITUDE
data_lat = data.LATITUDE
xx, yy = np.meshgrid(data_long, data_lat)

In [5]:
# from traning South Atlantic region
# weightsSeg = [0.02568828581339181, 0.46645665128011865, 0.5078550629064896]
weightsSeg = [0.04, 0.48, 0.48]

def dice_coef_anti(y_true, y_pred):
    smooth = 1.  # to avoid zero division
    y_true_anti = y_true[:,:,1]
    y_pred_anti = y_pred[:,:,1]
    intersection_anti = K.sum(y_true_anti * y_pred_anti)
    return (2 * intersection_anti + smooth) / (K.sum(y_true_anti)+ K.sum(y_pred_anti) + smooth)

def dice_coef_cyc(y_true, y_pred):
    smooth = 1.  # to avoid zero division
    y_true_cyc = y_true[:,:,2]
    y_pred_cyc = y_pred[:,:,2]
    intersection_cyc = K.sum(y_true_cyc * y_pred_cyc)
    return (2 * intersection_cyc + smooth) / (K.sum(y_true_cyc) + K.sum(y_pred_cyc) + smooth)

def dice_coef_nn(y_true, y_pred):
    smooth = 1.  # to avoid zero division
    y_true_nn = y_true[:,:,0]
    y_pred_nn = y_pred[:,:,0]
    intersection_nn = K.sum(y_true_nn * y_pred_nn)
    return (2 * intersection_nn + smooth) / (K.sum(y_true_nn) + K.sum(y_pred_nn) + smooth)
    
def mean_dice_coef(y_true, y_pred):
    return (dice_coef_anti(y_true, y_pred) + dice_coef_cyc(y_true, y_pred) + dice_coef_nn(y_true, y_pred))/3.

def weighted_mean_dice_coef(y_true, y_pred):
    return (weightsSeg[2]*dice_coef_anti(y_true, y_pred) + weightsSeg[1]*dice_coef_cyc(y_true, y_pred) + weightsSeg[0]*dice_coef_nn(y_true, y_pred))

def get_anti_f1_score(y_true, y_pred):
    return dice_coef_anti(y_true, y_pred)

def get_cyc_f1_score(y_true, y_pred):
    return dice_coef_cyc(y_true, y_pred)

def get_nn_f1_score(y_true, y_pred):
    return dice_coef_nn(y_true, y_pred)

def dice_coef_loss(y_true, y_pred):
    return 1 - weighted_mean_dice_coef(y_true, y_pred)

In [6]:
# accuracy = (TP+TN)/((TP+TN)+(FP+FN))
def accuracy_cyc(y_true, y_pred):
    y_true_cyc = y_true[:,:,2]
    y_pred_cyc = y_pred[:,:,2]
    correct_predictions = K.sum(K.cast(y_true_cyc == y_pred_cyc, tf.float32))
    total = K.sum(K.cast(y_true_cyc == y_pred_cyc, tf.float32)) + K.sum(K.cast(y_true_cyc != y_pred_cyc, tf.float32))
    return (correct_predictions/total)

def accuracy_anti(y_true, y_pred):
    y_true_anti = y_true[:,:,1]
    y_pred_anti = y_pred[:,:,1]
    correct_predictions = K.sum(K.cast(y_true_anti == y_pred_anti, tf.float32))
    total = K.sum(K.cast(y_true_anti == y_pred_anti, tf.float32)) + K.sum(K.cast(y_true_anti != y_pred_anti, tf.float32))
    return (correct_predictions/total)

def accuracy_nn(y_true, y_pred):
    y_true_nn = y_true[:,:,0]
    y_pred_nn = y_pred[:,:,0]
    correct_predictions = K.sum(K.cast(y_true_nn == y_pred_nn, tf.float32))
    total = K.sum(K.cast(y_true_nn == y_pred_nn, tf.float32)) + K.sum(K.cast(y_true_nn != y_pred_nn, tf.float32))
    return (correct_predictions/total)

## Model trained on South Atlantic region

In [7]:
img_size = (840, 480)
num_classes = 3
weight_path_south_atlantic = "/albedo/work/user/ssunar/for_paper/unet_trained/south_atlantic/train/weights.h5"
test_gen = plain_net_eddy(30, img_size, X_south_atlantic, test_y_south_atlantic)

model_south_atlantic = get_model(img_size, num_classes)
model_south_atlantic.load_weights(weight_path_south_atlantic)
model_south_atlantic.compile(
    loss=dice_coef_loss, 
    metrics=['categorical_accuracy', mean_dice_coef, weighted_mean_dice_coef, accuracy_cyc, accuracy_anti, accuracy_nn]
)
test_accuracy_south_atlantic = model_south_atlantic.evaluate(test_gen, verbose=0)
print(f"loss: {test_accuracy_south_atlantic[0]}")
print(f"categorical_accuracy: {test_accuracy_south_atlantic[1]}")
print(f"mean_dice_coef: {test_accuracy_south_atlantic[2]}")
print(f"weighted_mean_dice_coef: {test_accuracy_south_atlantic[3]}")

print(f"Accuracy Cyc: {test_accuracy_south_atlantic[4]}")
print(f"Accuracy Anti-cyc: {test_accuracy_south_atlantic[5]}")
print(f"Accuracy Background: {test_accuracy_south_atlantic[6]}")

2024-05-24 00:42:53.405831: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 43497 MB memory:  -> device: 0, name: NVIDIA A40, pci bus id: 0000:41:00.0, compute capability: 8.6
2024-05-24 00:42:57.105463: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:432] Loaded cuDNN version 8904
2024-05-24 00:42:58.620782: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:606] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


loss: 0.3839283883571625
categorical_accuracy: 0.918411135673523
mean_dice_coef: 0.7194523215293884
weighted_mean_dice_coef: 0.6160714626312256
Accuracy Cyc: 0.946439266204834
Accuracy Anti-cyc: 0.9094130992889404
Accuracy Background: 0.8684424757957458


## Model trained on Gulfstream region

In [8]:
weight_path_gulfstream = "/albedo/work/user/ssunar/for_paper/unet_trained/Gulfstream/train/weights.h5"

model_gulfstream = get_model(img_size, num_classes)
model_gulfstream.load_weights(weight_path_gulfstream)
model_gulfstream.compile(
    loss=dice_coef_loss, 
    metrics=['categorical_accuracy', mean_dice_coef, weighted_mean_dice_coef, accuracy_cyc, accuracy_anti, accuracy_nn]
)
test_accuracy_gulfstream = model_gulfstream.evaluate(test_gen, verbose=0)
print(f"loss: {test_accuracy_gulfstream[0]}")
print(f"categorical_accuracy: {test_accuracy_gulfstream[1]}")
print(f"mean_dice_coef: {test_accuracy_gulfstream[2]}")
print(f"weighted_mean_dice_coef: {test_accuracy_gulfstream[3]}")

print(f"Accuracy Cyc: {test_accuracy_gulfstream[4]}")
print(f"Accuracy Anti-cyc: {test_accuracy_gulfstream[5]}")
print(f"Accuracy Background: {test_accuracy_gulfstream[6]}")

loss: 0.39927002787590027
categorical_accuracy: 0.9123671054840088
mean_dice_coef: 0.7077065706253052
weighted_mean_dice_coef: 0.6007300019264221
Accuracy Cyc: 0.9366570711135864
Accuracy Anti-cyc: 0.9359309077262878
Accuracy Background: 0.8469423055648804
